In [2]:
%pip install duckdb
import pandas as pd
import duckdb

   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   --- ------------------------------------ 1.0/13.7 MB 9.5 MB/s eta 0:00:02
   --------- ------------------------------ 3.4/13.7 MB 11.1 MB/s eta 0:00:01
   ---------------- ----------------------- 5.8/13.7 MB 11.4 MB/s eta 0:00:01
   ----------------------- ---------------- 8.1/13.7 MB 11.5 MB/s eta 0:00:01
   ------------------------------ --------- 10.5/13.7 MB 11.6 MB/s eta 0:00:01
   ------------------------------------- -- 12.8/13.7 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 13.7/13.7 MB 10.8 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Consideremos o arquivo itr DRE como EXEMPLO

In [27]:
arq = "data/interim/itr_3_filtered/itr_cia_aberta_DRE_2011-2025.parquet"

Antes de qualquer análise, verifica-se a quantidade total de registros presentes na base para compreender sua dimensão.

In [28]:
duckdb.sql(f"""
SELECT
    COUNT(*) AS total_registros
FROM '{arq}'
""").df()

,total_registros
0,1991007


---
**Para prosseguir com a construção dos bancos de dados, precisamos encontrar uma forma de identificar uma empresa/companhia aberta de forma exclusiva.**

Para isso, precisamos entender quantos cnpjs, razões sociais e códigos cvm existem no banco de dados da cvm

In [29]:
duckdb.sql(f"""
SELECT
    COUNT(DISTINCT CNPJ_CIA) AS cnpjs
FROM '{arq}'
""").df()

,cnpjs
0,721


In [30]:
duckdb.sql(f"""
SELECT
    COUNT(DISTINCT DENOM_CIA) AS razoes_sociais
FROM '{arq}'
""").df()

,razoes_sociais
0,756


In [31]:
duckdb.sql(f"""
SELECT
    COUNT(DISTINCT CD_CVM) AS codigos_cvm
FROM '{arq}'
""").df()

,codigos_cvm
0,725


A quantidade de cnpjs, razões sociais e códigos cvm são diferentes! Por que?

- Razões sociais: Companhias podem mudar de razão social (nome) ao longo do tempo (fusões, aquisições, mudanças societárias, etc)
    - Assim é normal que haja mais razões sociais que cnpjs e códigos cvm

- E quanto ao cnpj e código cvm? Há 725 códigos cvm e 721 cnpjs, o que isso significa?

Existem mais de um CNPJ associado a um mesmo CÓDIGO CVM?

In [37]:
duckdb.sql(f"""
SELECT
    CD_CVM,
    COUNT(DISTINCT CNPJ_CIA) AS qtd_cnpjs
FROM '{arq}'
GROUP BY CD_CVM
HAVING COUNT(DISTINCT CNPJ_CIA) > 1
ORDER BY qtd_cnpjs DESC, CD_CVM
""").df()

,CD_CVM,qtd_cnpjs


Não!  
E existem mais de um CÓDIGO CVM associado a um mesmo CNPJ? (essa resposta deve ser, a priori, afirmativa, dado que há mais códigos cvm que cnpjs na base)

In [38]:
duckdb.sql(f"""
SELECT
    CNPJ_CIA,
    COUNT(DISTINCT CD_CVM) AS qtd_codigos
FROM '{arq}'
GROUP BY CNPJ_CIA
HAVING COUNT(DISTINCT CD_CVM) > 1
ORDER BY qtd_codigos DESC, CNPJ_CIA
""").df()

,CNPJ_CIA,qtd_codigos
0,08.926.302/0001-05,2
1,09.149.503/0001-06,2
2,59.717.553/0001-02,2
3,62.258.884/0001-36,2


Como concluimos, a priori, existe! 

Por que isso ocorre?
- Embora incomum, a reutilização do mesmo CNPJ em registros diferentes da CVM pode acontecer em reorganizações societárias ou mudanças de registro.

**A análise inicial indica que cada Código CVM está associado a apenas um CNPJ distinto. Entretanto, alguns CNPJs podem estar associados a mais de um Código CVM.**

Indepentemente dos porquês de haverem mais de um código cvm para um mesmo cnpj, normalmente bancos de dados financeiros são estruturados em relação ao código cvm.  
Todo o ecossistema da CVM gira em torno dele. O CNPJ é uma característica da companhia, já o CD_CVM é o identificador do registro regulatório.
 
Assim, **identificaremos uma companhia de forma exclusiva através de seu código cvm.**

---


Para construir o banco de dados, o identificador da companhia será o código cvm, mas como atribuímos o nome da companhia, considerando que pode haver mais de uma razão social para este mesmo código cvm?

Para a construção de uma tabela mestre de empresas, **será utilizada a razão social mais recente disponível na base**, enquanto as demonstrações financeiras preservarão a razão social registrada em cada data de referência.

---

Agora surge outro problema: **nem todas companhias listadas da cvm estão ou foram listadas na bolsa de valores**. Como identificar tais companhias?

As duas abordagens imediatas são:
- Utilizar os dados cadastrais de companhias abertas da cvm (https://dados.cvm.gov.br/dataset/cia_aberta-cad#:~:text=Cias%20Abertas%3A%20Informa%C3%A7%C3%A3o%20Cadastral.)
    - Banco de dados pronto com todas companhias abertas em cada período
    - Grande problema: Existem empresas de capital aberto registradas na CVM que nunca tiveram negociação relevante, empresas em recuperação judicial, companhias fechando capital, companhias listadas mas sem liquidez etc.
- Utilizar as séries históricas da B3 (COTAHIST) que exibem todas as transações negociadas de forma padronizada
    - se um ticker aparece no arquivo daquele ano, houve pelo menos uma negociação naquele ano.
    - é identificar o padrão dos arquivos textos e construir o banco de dados de ações negociadas em cada período

**Construiremos o banco de dados de empresas negociadas na bolsa em cada período através das séries históricas da B3**